# 07 A policy check after the gate: rules as data, a solver that names the rule

The gate decides *which route* a message takes. It does not decide whether Meridian may *act* on a refund claim: that is policy (days before departure, fare type, who cancelled),
and policy belongs in code a reviewer can read. Here the refund policy is a short list of **named rules**, and a solver (Z3) does two jobs that a pile of `if` statements does badly:

1. **Check a proposed action.** If the system wants to approve a full refund and a rule forbids it, the solver returns the **names of the rules that clash** (an unsat core).
2. **Check the rule list itself.** Does any case get two different outcomes (a conflict)? Does any case get none (a gap)? The solver returns a concrete example case.

**What this does not do.** It checks the rules someone wrote; it cannot tell whether they match the written policy. The facts (days to departure, fare paid) must come from a booking system
or from a reviewed extraction step, not from this notebook: here they are structured inputs. Z3 returns an unsat core that is not guaranteed to be minimal, so we name "rules involved", not "the one rule".
No model, no network and no API key is used in this notebook.

In [1]:
import json, time, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="pkg_resources")   # z3 4.12 imports it; harmless
from z3 import Solver, Bool, Int, And, Or, Not, Implies, sat, unsat, BoolVal
import meridian as m

RESULTS = Path("data/results")
print("hardware:", m.hardware())

hardware: Intel Xeon Processor (Skylake, IBRS, no TSX), 4 logical cores, Linux 6.8.0-87-generic


## The rules, as data
Facts: `days` (days before departure when the customer asks), `flex` (flexible fare), `airline` (the airline cancelled). Outcome: `full`, `partial` (50%) or `deny`.
Each rule is a name, a condition and the outcome it requires.

In [2]:
days, flex, airline = Int("days"), Bool("flex"), Bool("airline")
OUTCOMES = ["full", "partial", "deny"]
out = {o: Bool(f"out_{o}") for o in OUTCOMES}

POLICY = [
    ("R1 airline cancelled: refund in full",            airline,                                      "full"),
    ("R2 flexible fare, 2+ days out: refund in full",   And(Not(airline), flex, days >= 2),           "full"),
    ("R3 standard fare, 14+ days out: refund in full",  And(Not(airline), Not(flex), days >= 14),     "full"),
    ("R4 standard fare, 2 to 13 days out: half",        And(Not(airline), Not(flex), days >= 2, days < 14), "partial"),
    ("R5 under 2 days out, customer cancelled: none",   And(Not(airline), days < 2),                  "deny"),
]

def outcome_is(o):
    return And(*[out[x] == (x == o) for x in OUTCOMES])

def rule_formula(rule):
    _, cond, o = rule
    return Implies(cond, outcome_is(o))

## Job 1: check a proposed action and name the rules in the way
A case arrives with facts; the system (the encoder-and-LLM cascade, or a person) proposes an outcome. We assert every rule under its name, add the facts and the proposal, and ask Z3.

In [3]:
def check(facts: dict, proposed: str, policy=POLICY):
    s = Solver()
    for name, cond, o in policy:
        s.assert_and_track(rule_formula((name, cond, o)), name)
    s.add(days == facts["days"], flex == facts["flex"], airline == facts["airline"], outcome_is(proposed))
    t = time.perf_counter(); r = s.check(); ms = (time.perf_counter() - t) * 1000
    return ("allowed", [], ms) if r == sat else ("blocked", sorted(str(c) for c in s.unsat_core()), ms)

cases = [
    ("airline cancelled, full refund",             dict(days=1,  flex=False, airline=True),  "full"),
    ("standard fare, 5 days out, full refund",     dict(days=5,  flex=False, airline=False), "full"),
    ("standard fare, 5 days out, half refund",     dict(days=5,  flex=False, airline=False), "partial"),
    ("flexible fare, 5 days out, full refund",     dict(days=5,  flex=True,  airline=False), "full"),
    ("customer cancels the day before, full",      dict(days=1,  flex=True,  airline=False), "full"),
    ("customer cancels the day before, none",      dict(days=1,  flex=True,  airline=False), "deny"),
]
for label, facts, proposed in cases:
    verdict, core, ms = check(facts, proposed)
    print(f"{label:45s} -> {verdict:8s} {ms:5.1f} ms  {core}")

airline cancelled, full refund                -> allowed    0.6 ms  []
standard fare, 5 days out, full refund        -> blocked    0.4 ms  ['R4 standard fare, 2 to 13 days out: half']
standard fare, 5 days out, half refund        -> allowed    0.5 ms  []
flexible fare, 5 days out, full refund        -> allowed    0.4 ms  []
customer cancels the day before, full         -> blocked    0.4 ms  ['R5 under 2 days out, customer cancelled: none']
customer cancels the day before, none         -> allowed    0.5 ms  []


## Job 2: check the rule list itself
Two questions over *all* possible cases, not a sample. A **conflict** is a case where two rules require different outcomes. A **gap** is a case where no rule applies.
Z3 either proves there is none or hands back an example.

In [4]:
def conflicts(policy):
    found = []
    for i in range(len(policy)):
        for j in range(i + 1, len(policy)):
            (ni, ci, oi), (nj, cj, oj) = policy[i], policy[j]
            if oi == oj: continue
            s = Solver(); s.add(ci, cj, days >= 0, days <= 365)
            if s.check() == sat:
                mdl = s.model(); found.append((ni, nj, {str(d): str(mdl[d]) for d in mdl.decls()}))
    return found

def gaps(policy):
    s = Solver(); s.add(days >= 0, days <= 365, *[Not(cond) for _, cond, _ in policy])
    if s.check() == sat:
        mdl = s.model(); return {str(d): str(mdl[d]) for d in mdl.decls()}
    return None

print("current policy  conflicts:", conflicts(POLICY), "| gap:", gaps(POLICY))

# A plausible edit made in a hurry: the flexible-fare rule loses its "2+ days out" condition.
BUGGY = [r if not r[0].startswith("R2") else ("R2 flexible fare: refund in full", And(Not(airline), flex), "full") for r in POLICY]
print("after the edit  conflicts:")
for c in conflicts(BUGGY): print("  ", c)

# And a policy with a hole: the half-refund rule is deleted.
HOLE = [r for r in POLICY if not r[0].startswith("R4")]
print("rule R4 deleted gap example:", gaps(HOLE))

current policy  conflicts: [] | gap: None
after the edit  conflicts:
   ('R2 flexible fare: refund in full', 'R5 under 2 days out, customer cancelled: none', {'days': '0', 'airline': 'False', 'flex': 'True'})
rule R4 deleted gap example: {'days': '10', 'flex': 'False', 'airline': 'False'}


## Cost and agreement with plain code
For four short rules a Python function gives the same answers, so the solver has to earn its place. Here are both on 5,000 random cases: do they agree, and what does each cost per decision?

In [5]:
def plain(facts):
    if facts["airline"]: return "full"
    if facts["days"] < 2: return "deny"
    if facts["flex"] or facts["days"] >= 14: return "full"
    return "partial"

random.seed(m.SEED)
sample = [dict(days=random.randint(0, 60), flex=random.random() < 0.3, airline=random.random() < 0.15) for _ in range(5000)]
t = time.perf_counter(); expected = [plain(f) for f in sample]; plain_ms = (time.perf_counter() - t) * 1000 / len(sample)

agree, solver_ms = 0, []
for f, e in zip(sample, expected):
    ok = check(f, e)
    agree += ok[0] == "allowed"; solver_ms.append(ok[2])
wrong_blocked = sum(check(f, "full" if e != "full" else "deny")[0] == "allowed" for f, e in zip(sample[:500], expected[:500]))
print(f"solver allows the plain function's outcome on {agree} of {len(sample)} cases")
print(f"solver allows a *different* outcome on {wrong_blocked} of 500 cases (should be 0)")
print(f"median {sorted(solver_ms)[len(solver_ms)//2]:.2f} ms per check with the solver, {plain_ms*1000:.1f} microseconds per case in plain Python")

solver allows the plain function's outcome on 5000 of 5000 cases
solver allows a *different* outcome on 0 of 500 cases (should be 0)
median 0.37 ms per check with the solver, 0.2 microseconds per case in plain Python


In [6]:
res = dict(n_cases=len(sample), agree=agree, wrong_allowed=wrong_blocked,
           solver_median_ms=sorted(solver_ms)[len(solver_ms)//2], plain_us=plain_ms * 1000,
           current_conflicts=len(conflicts(POLICY)), edited_conflicts=conflicts(BUGGY), hole_example=gaps(HOLE),
           hardware=m.hardware(), z3_version=__import__("z3").get_version_string())
(RESULTS / "07_policy_check.json").write_text(json.dumps(res, indent=2, default=str))

604

## What to take away
- The solver and the plain function agree on every sampled case (the numbers are printed above), and the solver costs milliseconds per decision where the function costs microseconds. For a fixed list of five rules, write the function.
- The solver earns its place on Job 2: it examined every possible case, not a sample, and found the conflict the hurried edit introduced and the case nobody had covered, with a concrete example each time.
- The rules stay data: a reviewer reads five lines, and a block names the rules involved.
- Limits: the check is only as good as the rules and the facts. If the facts come from a model (the days to departure read out of an email), that extraction is a new error source to measure.